# FINAL BOSS — Build Your Own Doodle AI
### Choose your model. Train it. Try to fool it.

**Guided Mode:** Copy working ideas from the three Method notebooks.  
**Challenge Mode:** Change features, algorithms, and hyperparameters.

This is the ONLY notebook with TODO prompts. The default baseline is runnable immediately, so beginners can succeed within seven minutes.

**Rules:** Tune on validation. Test once at the end. Fresh human drawings are a fun demonstration, not a statistically reliable benchmark.

In [ ]:
# Shared setup: works from local Jupyter or a fresh Google Colab runtime.
import subprocess, sys
from pathlib import Path
here = Path.cwd().resolve()
candidates = [here, here.parent, here / "doodle-ml-workshop", here.parent / "doodle-ml-workshop"]
ROOT = next((p for p in candidates if (p / "src" / "data.py").exists()), None)
if ROOT is None:
    checkout = Path("/content/Computer-Vision-Workshop")
    if not checkout.exists():
        subprocess.run([
            "git", "clone", "--depth", "1", "--branch", "doodle-ai-workshop",
            "https://github.com/jaekop/Computer-Vision-Workshop.git", str(checkout)
        ], check=True)
    ROOT = checkout / "doodle-ml-workshop"
sys.path.insert(0, str(ROOT))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")])
print("Workshop ready at", ROOT)

## 1. Your data — already prepared
The training, validation and sealed test split are the same as the Method notebooks. Don't look at test predictions while selecting your model.

In [ ]:
from src.data import load_data
from src.models import fit_model
from src.evaluation import validation_accuracy, show_confusion, show_mistakes
import matplotlib.pyplot as plt
import sys

data = load_data(cache_dir=ROOT / "data")
print("Choose from:", ", ".join(data.class_names))
print("Training drawings:", len(data.y_train), "Validation drawings:", len(data.y_val))

## 2. TODO: choose your pipeline
**Bronze:** Run the working KNN baseline.  
**Silver:** Try at least one alternative and compare validation performance.  
**Gold:** Inspect a confusion matrix, explain a failure, and improve it.

These are **task milestones**, not invented accuracy percentages.

In [ ]:
# TODO 1: change these choices to customize your recognizer.
# Options: 'knn', 'tree', 'forest', 'svm'
MODEL_NAME = 'knn'
# Options: 'pixels', 'hog'
REPRESENTATION = 'pixels'
# Center the drawing before extracting features?
CENTER = False
# Examples: {'n_neighbors': 5}, {'max_depth': 12},
#           {'n_estimators': 80}, {'C': 1.0}
PARAMETERS = {'n_neighbors': 5}

model = fit_model(data.X_train, data.y_train, MODEL_NAME,
                  representation=REPRESENTATION, center=CENTER, **PARAMETERS)
print('Your model:', MODEL_NAME, '| features:', REPRESENTATION, '| centered:', CENTER)

## 3. Check your validation score
Use this score to decide what to try next. A model that performs worse still teaches you something.

In [ ]:
score = validation_accuracy(model, data.X_val, data.y_val)
print(f"Validation accuracy: {score:.3f}")
show_confusion(model, data.X_val, data.y_val, title="Your validation confusion matrix")
plt.show()

## 4. TODO: improve one thing
Try `MODEL_NAME = "forest"` or `MODEL_NAME = "svm"` with `REPRESENTATION = "hog"`. If you switch model types, update PARAMETERS to match that model.

**Write down:** What changed? What did you predict? What did the validation score show?

In [ ]:
# TODO 2: optional extra experiment — fully working example.
challenger = fit_model(data.X_train, data.y_train, 'forest',
                       representation='pixels', n_estimators=80)
challenger_score = validation_accuracy(challenger, data.X_val, data.y_val)
print(f'Baseline validation: {score:.3f}')
print(f'Challenger validation: {challenger_score:.3f}')
# To use the challenger for the final demo, uncomment the next line:
# model = challenger

## 5. Inspect mistakes before the final test
Can you explain at least one error? Is it a representation issue, a confusing shape, or an unfamiliar style?

In [ ]:
show_mistakes(model, data.X_val, data.y_val, count=12)
plt.show()

## 6. FINAL CHECK — reveal the sealed test score ONCE
Run this only after you have selected your pipeline using validation. Do not go back and tune based on the result.

In [ ]:
from sklearn.metrics import accuracy_score
test_score = accuracy_score(data.y_test, model.predict(data.X_test))
print(f"Final held-out TEST accuracy: {test_score:.3f}")
print("Do not tune your model based on this number.")

## 7. Launch your personal doodle playground
The app runs in **this notebook's runtime** and uses **your trained model**. In Colab, a temporary public Gradio share link may be created. Don't enter personal information in a shared demo.

Draw dark strokes on a light canvas. Use the preview to inspect what the classifier receives. The scores from a linear SVM are margins, not probabilities.

In [ ]:
from src.playground import launch_playground
IN_COLAB = "google.colab" in sys.modules
launch_playground({"My Doodle AI": model}, share=IN_COLAB)

## BONUS — Human vs Machine
Swap with another pair. Draw three to five new examples from the **six allowed classes** and see which predictions fail. Try an object outside those classes to expose a limitation, but do not interpret the output as reliable out-of-distribution detection.

### Reflection
1. Which feature representation helped most?
2. Which doodles are confused most often?
3. What data would you collect next?
4. Why is a one-time test split important?

**You built a classical ML image classifier.**